# trader_6 backtest

Goal: replay trader_6.py across days -2, -1, 0 with a taker-only fill model and grid-search three OSMIUM stop-loss trigger families. Rank by total PnL across all 3 days, pick the winner, and report params to paste back into `trader_6.py`.

Root cause (trader.py, log 169826): `fair_value = 0.5*10_000 + 0.5*ema_slow` stayed pinned near 10000 while OSMIUM drifted down to ~9996, so the sell-taker `bid_price < fair_value` gate blocked hits and the maker kept bidding 9999 into a falling tape with +80 inventory. Fix in trader_6 is regime-aware `fair`, inventory skew on `ref`, plus a hard-flush stop-loss trigger (tuned below).

In [16]:
import sys, importlib, io, json, math, copy, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('/Users/devansharora/Desktop/Trading_System/IMC_Prosperity')
R1 = ROOT / 'Round_1' / 'ROUND1'
sys.path.insert(0, str(R1))

import datamodel
importlib.reload(datamodel)
from datamodel import OrderDepth, TradingState, Order

import trader_6
importlib.reload(trader_6)

PEPPER = trader_6.PEPPER
OSMIUM = trader_6.OSMIUM
POSITION_LIMIT = trader_6.POSITION_LIMIT
print('loaded', trader_6.__file__)

loaded /Users/devansharora/Desktop/Trading_System/IMC_Prosperity/Round_1/ROUND1/trader_6.py


In [17]:
# ── load 3 days of prices ────────────────────────────────────────────
frames = []
for d in (-2, -1, 0):
    p = R1 / f'prices_round_1_day_{d}.csv'
    df = pd.read_csv(p)
    df['day'] = d
    frames.append(df)
prices = pd.concat(frames, ignore_index=True)
prices = prices.sort_values(['day', 'timestamp']).reset_index(drop=True)
print(prices.shape, prices['product'].unique(), prices['day'].unique())
prices.head()

(60000, 17) <ArrowStringArray>
['INTARIAN_PEPPER_ROOT', 'ASH_COATED_OSMIUM']
Length: 2, dtype: str [-2 -1  0]


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,-2,0,INTARIAN_PEPPER_ROOT,9992.0,17.0,NaN,NaN,NaN,NaN,10005.0,9.0,10008.0,17.0,NaN,NaN,9998.5,0.0
1,-2,0,ASH_COATED_OSMIUM,NaN,NaN,NaN,NaN,NaN,NaN,10010.0,25.0,NaN,NaN,NaN,NaN,10010.0,0.0
2,-2,100,INTARIAN_PEPPER_ROOT,9995.0,11.0,9992.0,16.0,NaN,NaN,10006.0,11.0,10008.0,16.0,NaN,NaN,10000.5,0.0
3,-2,100,ASH_COATED_OSMIUM,9992.0,15.0,NaN,NaN,NaN,NaN,10008.0,15.0,10011.0,20.0,NaN,NaN,10000.0,0.0
4,-2,200,INTARIAN_PEPPER_ROOT,9995.0,12.0,NaN,NaN,NaN,NaN,10008.0,20.0,NaN,NaN,NaN,NaN,10001.5,0.0


In [18]:
# ── build per-tick order book snapshots ──────────────────────────────
def row_to_depth(row):
    depth = OrderDepth()
    for k in (1, 2, 3):
        bp = row.get(f'bid_price_{k}')
        bv = row.get(f'bid_volume_{k}')
        if pd.notna(bp) and pd.notna(bv) and bv > 0:
            depth.buy_orders[int(bp)] = int(bv)
        ap = row.get(f'ask_price_{k}')
        av = row.get(f'ask_volume_{k}')
        if pd.notna(ap) and pd.notna(av) and av > 0:
            depth.sell_orders[int(ap)] = -int(av)
    return depth

# group by (day, timestamp) -> {product: depth}
def build_ticks(df):
    ticks = []
    for (day, ts), g in df.groupby(['day', 'timestamp'], sort=True):
        depths = {}
        for _, row in g.iterrows():
            depths[row['product']] = row_to_depth(row)
        ticks.append((day, ts, depths))
    return ticks

ticks_by_day = {d: build_ticks(prices[prices['day'] == d]) for d in (-2, -1, 0)}
for d in (-2, -1, 0):
    print(f'day {d}: {len(ticks_by_day[d])} ticks')

day -2: 10000 ticks
day -1: 10000 ticks
day 0: 10000 ticks


In [19]:
# ── replay harness with taker-only fill model ───────────────────────
def simulate(Trader_cls, ticks, overrides=None):
    """Replay a full day. overrides: dict of attribute names to patch on trader_6 module.

    Returns dict with final PnL (cash + position * last_mid) per product, plus per-tick trace.
    Fill model: our BUY @ P fills min(our_qty, ask_vol at P') for all ask levels P' <= P.
                our SELL @ P fills min(our_qty, bid_vol at P') for all bid levels P' >= P.
                Passive (maker) quotes assumed un-filled (conservative, matches user choice).
    """
    # apply overrides
    saved = {}
    if overrides:
        for k, v in overrides.items():
            saved[k] = getattr(trader_6, k)
            setattr(trader_6, k, v)
    try:
        trader = Trader_cls()
        trader_data = ''
        position = {PEPPER: 0, OSMIUM: 0}
        cash = {PEPPER: 0.0, OSMIUM: 0.0}
        last_mid = {PEPPER: None, OSMIUM: None}
        pnl_trace = []
        for day, ts, depths in ticks:
            state = TradingState(
                traderData=trader_data,
                timestamp=ts,
                listings={},
                order_depths=depths,
                own_trades={},
                market_trades={},
                position=dict(position),
                observations=None,
            )
            # deep-copy depths so trader can't mutate simulator state
            result, _, trader_data = trader.run(state)
            for product, orders in result.items():
                depth = depths.get(product)
                if depth is None:
                    continue
                # snapshot asks / bids sorted by price
                asks = sorted([(p, abs(v)) for p, v in depth.sell_orders.items()])
                bids = sorted([(p, abs(v)) for p, v in depth.buy_orders.items()], reverse=True)
                for o in orders:
                    qty = o.quantity
                    if qty > 0:  # BUY @ o.price
                        need = qty
                        # only cross; don't fill maker bids
                        new_asks = []
                        for (ap, av) in asks:
                            if need <= 0 or ap > o.price:
                                new_asks.append((ap, av))
                                continue
                            fill = min(need, av)
                            position[product] += fill
                            cash[product] -= fill * ap
                            need -= fill
                            av -= fill
                            if av > 0:
                                new_asks.append((ap, av))
                        asks = new_asks
                    elif qty < 0:  # SELL @ o.price, qty negative
                        need = -qty
                        new_bids = []
                        for (bp, bv) in bids:
                            if need <= 0 or bp < o.price:
                                new_bids.append((bp, bv))
                                continue
                            fill = min(need, bv)
                            position[product] -= fill
                            cash[product] += fill * bp
                            need -= fill
                            bv -= fill
                            if bv > 0:
                                new_bids.append((bp, bv))
                        bids = new_bids
            # mark to market using mid of original snapshot
            for prod, depth in depths.items():
                bb = max(depth.buy_orders) if depth.buy_orders else None
                ba = min(depth.sell_orders) if depth.sell_orders else None
                if bb is not None and ba is not None:
                    last_mid[prod] = (bb + ba) / 2
                elif bb is not None:
                    last_mid[prod] = bb
                elif ba is not None:
                    last_mid[prod] = ba
            pnl_trace.append({
                'day': day, 'ts': ts,
                'pos_pepper': position[PEPPER], 'pos_osmium': position[OSMIUM],
                'mid_osmium': last_mid[OSMIUM],
                'pnl_osmium': cash[OSMIUM] + (position[OSMIUM] * last_mid[OSMIUM] if last_mid[OSMIUM] else 0),
                'pnl_pepper': cash[PEPPER] + (position[PEPPER] * last_mid[PEPPER] if last_mid[PEPPER] else 0),
            })
        pnl_osmium = cash[OSMIUM] + (position[OSMIUM] * last_mid[OSMIUM] if last_mid[OSMIUM] else 0)
        pnl_pepper = cash[PEPPER] + (position[PEPPER] * last_mid[PEPPER] if last_mid[PEPPER] else 0)
        return {
            'pnl_osmium': pnl_osmium,
            'pnl_pepper': pnl_pepper,
            'pnl_total': pnl_osmium + pnl_pepper,
            'trace': pd.DataFrame(pnl_trace),
        }
    finally:
        for k, v in saved.items():
            setattr(trader_6, k, v)

# sanity: run defaults on day -1
r = simulate(trader_6.Trader, ticks_by_day[-1])
print('defaults day -1:', {k: round(r[k], 2) for k in ('pnl_pepper', 'pnl_osmium', 'pnl_total')})

defaults day -1: {'pnl_pepper': 79192.0, 'pnl_osmium': 4861.0, 'pnl_total': 84053.0}


In [20]:
# ── baseline: trader.py (as reference) ──────────────────────────────
import trader as trader_v1
importlib.reload(trader_v1)

def simulate_ref(Trader_cls, ticks):
    trader = Trader_cls()
    trader_data = ''
    position = {PEPPER: 0, OSMIUM: 0}
    cash = {PEPPER: 0.0, OSMIUM: 0.0}
    last_mid = {PEPPER: None, OSMIUM: None}
    for day, ts, depths in ticks:
        state = TradingState('', ts, {}, depths, {}, {}, dict(position), None)
        result, _, trader_data = trader.run(state)
        for product, orders in result.items():
            depth = depths.get(product)
            if depth is None:
                continue
            asks = sorted([(p, abs(v)) for p, v in depth.sell_orders.items()])
            bids = sorted([(p, abs(v)) for p, v in depth.buy_orders.items()], reverse=True)
            for o in orders:
                qty = o.quantity
                if qty > 0:
                    need = qty
                    new_asks = []
                    for (ap, av) in asks:
                        if need <= 0 or ap > o.price:
                            new_asks.append((ap, av)); continue
                        f = min(need, av)
                        position[product] += f; cash[product] -= f * ap
                        need -= f; av -= f
                        if av > 0: new_asks.append((ap, av))
                    asks = new_asks
                elif qty < 0:
                    need = -qty
                    new_bids = []
                    for (bp, bv) in bids:
                        if need <= 0 or bp < o.price:
                            new_bids.append((bp, bv)); continue
                        f = min(need, bv)
                        position[product] -= f; cash[product] += f * bp
                        need -= f; bv -= f
                        if bv > 0: new_bids.append((bp, bv))
                    bids = new_bids
        for prod, depth in depths.items():
            bb = max(depth.buy_orders) if depth.buy_orders else None
            ba = min(depth.sell_orders) if depth.sell_orders else None
            if bb is not None and ba is not None: last_mid[prod] = (bb+ba)/2
            elif bb is not None: last_mid[prod] = bb
            elif ba is not None: last_mid[prod] = ba
    po = cash[OSMIUM] + (position[OSMIUM] * last_mid[OSMIUM] if last_mid[OSMIUM] else 0)
    pp = cash[PEPPER] + (position[PEPPER] * last_mid[PEPPER] if last_mid[PEPPER] else 0)
    return {'pnl_osmium': po, 'pnl_pepper': pp, 'pnl_total': po+pp}

ref_rows = []
for d in (-2, -1, 0):
    r = simulate_ref(trader_v1.Trader, ticks_by_day[d])
    r['day'] = d; ref_rows.append(r)
ref_df = pd.DataFrame(ref_rows)
print('trader.py reference:')
print(ref_df)
print('trader.py total:', ref_df['pnl_total'].sum())

trader.py reference:
   pnl_osmium  pnl_pepper  pnl_total  day
0      1605.0     79591.0    81196.0   -2
1      2920.0     79261.0    82181.0   -1
2      3174.0     79450.0    82624.0    0
trader.py total: 246001.0


In [21]:
# ── grid search: 3 trigger families + 'none' baseline ───────────────
def run_combo(overrides):
    rows = []
    for d in (-2, -1, 0):
        r = simulate(trader_6.Trader, ticks_by_day[d], overrides=overrides)
        r_out = {'day': d,
                 'pnl_osmium': r['pnl_osmium'],
                 'pnl_pepper': r['pnl_pepper'],
                 'pnl_total': r['pnl_total']}
        # peak drawdown in osmium cumulative pnl (reporting only)
        trace = r['trace']
        if len(trace) > 0:
            running_max = trace['pnl_osmium'].cummax()
            dd = (trace['pnl_osmium'] - running_max).min()
            r_out['max_dd_osmium'] = dd
        rows.append(r_out)
    df = pd.DataFrame(rows)
    return df

combos = []
# baseline: no flush
combos.append(('none', {'OSMIUM_FLUSH_MODE': 'none'}))
# family 1: ema_cross_consec
for c in (2, 3, 4, 5):
    combos.append((f'ema_cross_consec_c{c}',
                   {'OSMIUM_FLUSH_MODE': 'ema_cross_consec', 'OSMIUM_FLUSH_CONSEC': c}))
# family 2: rolling_drop
for w in (20, 40, 60):
    for dr in (2, 3, 4):
        combos.append((f'rolling_drop_w{w}_d{dr}',
                       {'OSMIUM_FLUSH_MODE': 'rolling_drop', 'OSMIUM_FLUSH_W': w, 'OSMIUM_FLUSH_DROP': dr}))
# family 3: zscore_neg
for w in (20, 40, 60):
    for z in (1.5, 2.0, 2.5):
        combos.append((f'zscore_neg_w{w}_z{z}',
                       {'OSMIUM_FLUSH_MODE': 'zscore_neg', 'OSMIUM_FLUSH_W': w, 'OSMIUM_FLUSH_Z': z}))

print(f'running {len(combos)} combos...')
t0 = time.time()
all_rows = []
for name, ov in combos:
    res = run_combo(ov)
    total_osmium = res['pnl_osmium'].sum()
    total_pepper = res['pnl_pepper'].sum()
    total = res['pnl_total'].sum()
    dd = res['max_dd_osmium'].min() if 'max_dd_osmium' in res else None
    all_rows.append({
        'combo': name,
        'mode': ov.get('OSMIUM_FLUSH_MODE'),
        'w': ov.get('OSMIUM_FLUSH_W'),
        'consec': ov.get('OSMIUM_FLUSH_CONSEC'),
        'drop': ov.get('OSMIUM_FLUSH_DROP'),
        'z': ov.get('OSMIUM_FLUSH_Z'),
        'osmium_total': total_osmium,
        'pepper_total': total_pepper,
        'pnl_total': total,
        'worst_dd_osmium': dd,
    })
print(f'done in {time.time()-t0:.1f}s')
results = pd.DataFrame(all_rows).sort_values('pnl_total', ascending=False).reset_index(drop=True)
pd.set_option('display.max_rows', 60)
results

running 23 combos...
done in 7.0s


,combo,mode,w,consec,drop,z,osmium_total,pepper_total,pnl_total,worst_dd_osmium
0,none,none,NaN,NaN,NaN,NaN,13551.5,238054.0,251605.5,-1788.0
1,rolling_drop_w60_d3,rolling_drop,60.0,NaN,3.0,NaN,13551.5,238054.0,251605.5,-1788.0
2,zscore_neg_w60_z2.0,zscore_neg,60.0,NaN,NaN,2.0,13551.5,238054.0,251605.5,-1788.0
3,zscore_neg_w60_z1.5,zscore_neg,60.0,NaN,NaN,1.5,13551.5,238054.0,251605.5,-1788.0
4,zscore_neg_w40_z2.5,zscore_neg,40.0,NaN,NaN,2.5,13551.5,238054.0,251605.5,-1788.0
5,zscore_neg_w40_z2.0,zscore_neg,40.0,NaN,NaN,2.0,13551.5,238054.0,251605.5,-1788.0
6,zscore_neg_w40_z1.5,zscore_neg,40.0,NaN,NaN,1.5,13551.5,238054.0,251605.5,-1788.0
7,zscore_neg_w20_z2.5,zscore_neg,20.0,NaN,NaN,2.5,13551.5,238054.0,251605.5,-1788.0
8,zscore_neg_w20_z2.0,zscore_neg,20.0,NaN,NaN,2.0,13551.5,238054.0,251605.5,-1788.0
9,zscore_neg_w20_z1.5,zscore_neg,20.0,NaN,NaN,1.5,13551.5,238054.0,251605.5,-1788.0


In [22]:
# ── pick winner by total PnL ─────────────────────────────────────────
winner = results.iloc[0]
print('WINNER:', winner['combo'])
print(winner)
print()
print('Paste into trader_6.py defaults:')
mode = winner['mode']
print(f'OSMIUM_FLUSH_MODE = "{mode}"')
if mode == 'ema_cross_consec':
    print(f'OSMIUM_FLUSH_CONSEC = {int(winner["consec"])}')
elif mode == 'rolling_drop':
    print(f'OSMIUM_FLUSH_W = {int(winner["w"])}')
    print(f'OSMIUM_FLUSH_DROP = {int(winner["drop"])}')
elif mode == 'zscore_neg':
    print(f'OSMIUM_FLUSH_W = {int(winner["w"])}')
    print(f'OSMIUM_FLUSH_Z = {winner["z"]}')
# keep results around for memory.md
results.head(10)

WINNER: none
combo                  none
mode                   none
w                       NaN
consec                  NaN
drop                    NaN
z                       NaN
osmium_total        13551.5
pepper_total       238054.0
pnl_total          251605.5
worst_dd_osmium     -1788.0
Name: 0, dtype: object

Paste into trader_6.py defaults:
OSMIUM_FLUSH_MODE = "none"


,combo,mode,w,consec,drop,z,osmium_total,pepper_total,pnl_total,worst_dd_osmium
0,none,none,NaN,NaN,NaN,NaN,13551.5,238054.0,251605.5,-1788.0
1,rolling_drop_w60_d3,rolling_drop,60.0,NaN,3.0,NaN,13551.5,238054.0,251605.5,-1788.0
2,zscore_neg_w60_z2.0,zscore_neg,60.0,NaN,NaN,2.0,13551.5,238054.0,251605.5,-1788.0
3,zscore_neg_w60_z1.5,zscore_neg,60.0,NaN,NaN,1.5,13551.5,238054.0,251605.5,-1788.0
4,zscore_neg_w40_z2.5,zscore_neg,40.0,NaN,NaN,2.5,13551.5,238054.0,251605.5,-1788.0
5,zscore_neg_w40_z2.0,zscore_neg,40.0,NaN,NaN,2.0,13551.5,238054.0,251605.5,-1788.0
6,zscore_neg_w40_z1.5,zscore_neg,40.0,NaN,NaN,1.5,13551.5,238054.0,251605.5,-1788.0
7,zscore_neg_w20_z2.5,zscore_neg,20.0,NaN,NaN,2.5,13551.5,238054.0,251605.5,-1788.0
8,zscore_neg_w20_z2.0,zscore_neg,20.0,NaN,NaN,2.0,13551.5,238054.0,251605.5,-1788.0
9,zscore_neg_w20_z1.5,zscore_neg,20.0,NaN,NaN,1.5,13551.5,238054.0,251605.5,-1788.0


In [23]:
# ── per-day breakdown for the winning combo (for memory.md) ──────────
winner_ov = {'OSMIUM_FLUSH_MODE': winner['mode']}
if winner['mode'] == 'ema_cross_consec':
    winner_ov['OSMIUM_FLUSH_CONSEC'] = int(winner['consec'])
elif winner['mode'] == 'rolling_drop':
    winner_ov['OSMIUM_FLUSH_W'] = int(winner['w'])
    winner_ov['OSMIUM_FLUSH_DROP'] = int(winner['drop'])
elif winner['mode'] == 'zscore_neg':
    winner_ov['OSMIUM_FLUSH_W'] = int(winner['w'])
    winner_ov['OSMIUM_FLUSH_Z'] = float(winner['z'])

detail = run_combo(winner_ov)
print('per-day PnL (winner):')
print(detail.to_string(index=False))

per-day PnL (winner):
 day  pnl_osmium  pnl_pepper  pnl_total  max_dd_osmium
  -2      3710.5     79543.0    83253.5        -1628.0
  -1      4861.0     79192.0    84053.0        -1554.0
   0      4980.0     79319.0    84299.0        -1788.0
